## Imports & Configuration

In [ ]:
pip install OpenAI

In [1]:
# Imports
import pandas as pd
from openai import OpenAI
import os
import ast
import json

In [2]:
#Access API key
API_KEY_FILE = "/Users/andre.chan/OneDrive - OneWorkplace/Tribal VW/Ad Hoc/Language Analysis/ModelGPT Analysis/API Key/omcpmg_gpt_key 3.csv"
KEY_DF  = pd.read_csv(API_KEY_FILE)
GPT_API_KEY = KEY_DF["opmg_gpt_api_key"].values[0]
GPT_API_KEY = str(GPT_API_KEY).strip().replace('\xa0', '').replace('\u00a0', '')
OPENAI_CLIENT = OpenAI(api_key=GPT_API_KEY)
GPT_MODEL = "gpt-5.2"

## Bare Bones Classification

In [3]:
# Read the dataset
df = pd.read_excel(r"C:\Users\andre.chan\OneDrive - OneWorkplace\Tribal VW\Ad Hoc\Language Analysis\ModelGPT Analysis\modelgpt_output_id3_id7_troc_q3.xlsx")
print(df.shape)
df.head()

(17922, 9)


,id,timestamp,name,userId,sessionId,tags,input,output,model
0,4781019c-c929-4a1b-a719-3239a648b527,2026-05-31 23:17:28.519,Chat Conversation for model 30702,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,NaN,"[""other""]",What colours are available?,The T-Roc is available in a variety of attract...,T-Roc
1,05cd57ff-e9cb-47e6-82ba-c27882f5b569,2026-07-01 17:35:28.495,Chat Conversation for model 30702,b7120856-5569-4aaa-ab02-bf943c7d2613,NaN,"[""chat-model: 30702"",""color""]",Green,User is asking about car colours,T-Roc
2,0708a3a7-3678-46b6-bc18-1290fd4707d9,2026-07-01 13:09:50.807,Chat Conversation for model 30275,235842df-ebed-401d-851b-349a9ff3db28,NaN,"[""car_range"",""chat-model: 30275""]",What's the ID.3 range?,I can help you with the range for the ID.3! Th...,ID.3
3,074bd0a6-3cc9-4c54-b344-ef7ad1b9af7c,2026-07-01 19:21:55.252,Chat Conversation for model 30702,4df9ab21-95c9-4846-84f4-af6d9567419b,NaN,"[""chat-model: 30702"",""color""]",Show me the colours,User is asking about car colours,T-Roc
4,07744113-5e36-45c3-b860-890ff9b48f38,2026-07-01 19:38:08.022,Chat Conversation for model 30702,b970ba93-7838-44cf-b5b9-673b306244fc,NaN,"[""other""]",Which versions of the T-Roc are available?,The new T-Roc is a new-generation compact SUV ...,T-Roc


In [4]:
# ── Load Agent Instructions ─────────────────────────────────────────────────── 
def load_instructions(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()
    
CLASSIFIER_INSTRUCTIONS = load_instructions(r"C:\Users\andre.chan\OneDrive - OneWorkplace\Tribal VW\Ad Hoc\Language Analysis\ModelGPT Analysis\ModelGPT Classification Instructions Final.md")
CLASSIFIER_INSTRUCTIONS = ''.join(c if ord(c) < 128 else ' ' for c in CLASSIFIER_INSTRUCTIONS) #safety for unregcognised characters that were appearing

In [5]:
def clean_text(text): #safety for unregcognised characters that were appearing
    if not isinstance(text, str):
        text = str(text)
    return text.replace('\xa0', ' ').replace('\u00a0', ' ')

# Make the LLM call/Classification function
def classify_user_query(user_inquiry: str) -> list:
    """
    Call the OpenAI API with the classification instruction file as system instruction.
    Return a list of 3 elements ["category", "intent","sentiment"].
    """
    user_inquiry = clean_text(user_inquiry)
    user_inquiry = ''.join(c if ord(c) < 128 else ' ' for c in user_inquiry)

    response = OPENAI_CLIENT.chat.completions.create(
        model=GPT_MODEL,
        temperature=0.1,
        messages=[
            {"role": "system", "content": CLASSIFIER_INSTRUCTIONS},
            {"role": "user",   "content": user_inquiry},
        ],
    )
    
    raw = response.choices[0].message.content.strip()
    
    try:
        response_content = json.loads(raw)
        
        if len(response_content) != 3:
            raise ValueError(f"Expected 3 items, got {len(response_content)}")
        
        print(f"{user_inquiry}: {response_content[0]} | {response_content[1]} | {response_content[2]}")
        return response_content

    except (ValueError, json.JSONDecodeError) as e:
        print(f"PARSE ERROR on input: {user_inquiry}")
        print(f"Raw response was: {repr(raw)}")
        return ["Unknown", "Unknown", "Unknown"]

In [6]:
# Apply the classification function to the dataset
# Apply it to the textual column (probably your "input" col) you'd lke to classify a category and sentiment

df[["Category", "Intent", "Sentiment"]] = df["input"].apply(classify_user_query).tolist()
df.to_excel('modelgpt_output_classified_test_q3.xlsx', index=False)

What colours are available?: Features | Research | neutral
Green: Features | Research | neutral
What's the ID.3 range?: Range & Mileage | Research | neutral
Show me the colours : Features | Research | neutral
Which versions of the T-Roc are available?: Trims | Research | neutral
What colours are available?: Features | Research | neutral
Is there any finance offers for the ID.3?: Price & Finance | Consideration | neutral
If I bought 1.5 e TSi 150 R-Line 5dr auto how much would heads up cost: Features | Consideration | neutral
What colours are available?: Features | Research | neutral
Boot space: Size & Capacity | Research | neutral
What colours are available?: Features | Research | neutral
Tell me about id3: General Info | Research | neutral
Which versions of the T-Roc are available?: Trims | Research | neutral
What is the price of a new T-Roc?: Price & Finance | Consideration | neutral
What's the ID.3 range?: Range & Mileage | Research | neutral
What's the ID.3 range?: Range & Mileage 

In [ ]:
# test = df.head(10).copy()

# test[["Category", "Intent", "Sentiment"]] = (
#     test["input"]
#     .apply(classify_user_query)
#     .tolist()
# )

# test

In [ ]:
# Read the combined dataset
# df_combined = pd.read_excel(r"C:\Users\andre.chan\OneDrive - OneWorkplace\Tribal VW\Ad Hoc\Language Analysis\ModelGPT Analysis\modelgpt_output_classified_combined.xlsx")
# print(df_combined.shape)
# df_combined.head()

In [8]:
#Add date column for groupings below in script
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df

,id,timestamp,name,userId,sessionId,tags,input,output,model,Category,Intent,Sentiment,date
0,4781019c-c929-4a1b-a719-3239a648b527,2026-05-31 23:17:28.519,Chat Conversation for model 30702,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,NaN,"[""other""]",What colours are available?,The T-Roc is available in a variety of attract...,T-Roc,Features,Research,neutral,2026-05-31
1,05cd57ff-e9cb-47e6-82ba-c27882f5b569,2026-07-01 17:35:28.495,Chat Conversation for model 30702,b7120856-5569-4aaa-ab02-bf943c7d2613,NaN,"[""chat-model: 30702"",""color""]",Green,User is asking about car colours,T-Roc,Features,Research,neutral,2026-07-01
2,0708a3a7-3678-46b6-bc18-1290fd4707d9,2026-07-01 13:09:50.807,Chat Conversation for model 30275,235842df-ebed-401d-851b-349a9ff3db28,NaN,"[""car_range"",""chat-model: 30275""]",What's the ID.3 range?,I can help you with the range for the ID.3! Th...,ID.3,Range & Mileage,Research,neutral,2026-07-01
3,074bd0a6-3cc9-4c54-b344-ef7ad1b9af7c,2026-07-01 19:21:55.252,Chat Conversation for model 30702,4df9ab21-95c9-4846-84f4-af6d9567419b,NaN,"[""chat-model: 30702"",""color""]",Show me the colours,User is asking about car colours,T-Roc,Features,Research,neutral,2026-07-01
4,07744113-5e36-45c3-b860-890ff9b48f38,2026-07-01 19:38:08.022,Chat Conversation for model 30702,b970ba93-7838-44cf-b5b9-673b306244fc,NaN,"[""other""]",Which versions of the T-Roc are available?,The new T-Roc is a new-generation compact SUV ...,T-Roc,Trims,Research,neutral,2026-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17917,c4b51367-57e9-489d-b7ae-e73e08533b70,2026-08-25 06:46:14.446,Chat Conversation for model 30702,8183ee15-3873-4c6f-9e7f-4e4ff03875b5,NaN,"[""other""]",Which versions of the T-Roc are available?,The new T-Roc is a new-generation compact SUV ...,T-Roc,Trims,Research,neutral,2026-08-25
17918,c544e3bc-bede-416d-b3e0-e561ed410004,2026-08-25 06:46:49.078,Chat Conversation for model 30702,8183ee15-3873-4c6f-9e7f-4e4ff03875b5,NaN,"[""chat-model: 30702"",""other""]",what versions of t-Roc are available as 4x4 or...,I'm afraid none of the T-Roc models currently ...,T-Roc,Trims,Research,neutral,2026-08-25
17919,d01692de-efac-41c6-9779-57216540600c,2026-08-25 06:55:26.944,Chat Conversation for model 30702,c07599f7-889a-4ca1-8ceb-cd3e33904047,NaN,"[""chat-model: 30702"",""other""]",Tell me about the new T-Roc?,The new T-Roc is our stylish and versatile SUV...,T-Roc,General Info,Research,neutral,2026-08-25
17920,d40969ad-96a1-45fb-b395-b45bf8f5c0ac,2026-08-25 09:11:52.685,Chat Conversation for model 30702,de553cd4-32ec-4a56-a4d6-19cc8fb79f86,NaN,"[""chat-model: 30702"",""other""]","t,-roc r line",The **T-Roc R-Line** is the top trim in the T-...,T-Roc,Trims,Research,neutral,2026-08-25


In [9]:
chat_prompt_count = (df.groupby(["date", "userId", "model"]).size().reset_index(name="input_count"))
# chat_prompt_count.to_excel('prompt_count.xlsx', index=False)

In [10]:
category_summary = (
    df.groupby(["date", "userId", "model", "Category"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
category_summary


Category,date,userId,model,Availability,Comparison,EV Considerations,Features,General Info,Government Grant,Other/Unclassifiable,"Ownership (Brochure, Motability, Servicing, Warranty)",Parking & Driver Assistance,Price & Finance,Range & Mileage,Size & Capacity,Speak to Agent,Technical Specs,Trims,cannot classify
0,2026-05-31,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,T-Roc,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,2026-06-01,03bac7a8-e3af-4754-a22c-98f80f83544e,T-Roc,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,2026-06-01,052dfc8b-0eb9-4822-8935-0293bd7fc8b8,T-Roc,0,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0
3,2026-06-01,061a7ff6-fd75-423f-b3b9-c423f6b42692,ID.3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,2026-06-01,09e06ecd-e461-4b7f-995d-d628c9cf4c90,ID.3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13892,2026-08-31,f6c38a66-4077-4ed1-a3b8-7063e2c33294,T-Roc,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
13893,2026-08-31,f767ee45-e060-4d0a-9056-933cf896fbf8,T-Roc,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
13894,2026-08-31,f7bb7b4f-fc61-4863-90e9-d841e978dee4,ID.3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
13895,2026-08-31,fac014e7-f899-4504-aac7-a7dbb99a6786,T-Roc,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [11]:
intent_summary = (
    df.groupby(["date", "userId", "model", "Intent"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
intent_summary

Intent,date,userId,model,Consideration,General Info,Post-purchase,Research,VW Brand Awareness,cannot classify
0,2026-05-31,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,T-Roc,0,0,0,1,0,0
1,2026-06-01,03bac7a8-e3af-4754-a22c-98f80f83544e,T-Roc,1,0,0,0,0,0
2,2026-06-01,052dfc8b-0eb9-4822-8935-0293bd7fc8b8,T-Roc,0,0,0,3,0,0
3,2026-06-01,061a7ff6-fd75-423f-b3b9-c423f6b42692,ID.3,1,0,0,0,0,0
4,2026-06-01,09e06ecd-e461-4b7f-995d-d628c9cf4c90,ID.3,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
13892,2026-08-31,f6c38a66-4077-4ed1-a3b8-7063e2c33294,T-Roc,0,0,0,1,0,0
13893,2026-08-31,f767ee45-e060-4d0a-9056-933cf896fbf8,T-Roc,0,0,0,1,0,0
13894,2026-08-31,f7bb7b4f-fc61-4863-90e9-d841e978dee4,ID.3,0,0,0,1,0,0
13895,2026-08-31,fac014e7-f899-4504-aac7-a7dbb99a6786,T-Roc,0,0,0,1,0,0


In [12]:
sentiment_summary = (
    df.groupby(["date", "userId", "model", "Sentiment"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
sentiment_summary

Sentiment,date,userId,model,negative,neutral,no sentiment,positive
0,2026-05-31,3b6428ce-5e6b-40de-8a5f-b1d5054ca13b,T-Roc,0,1,0,0
1,2026-06-01,03bac7a8-e3af-4754-a22c-98f80f83544e,T-Roc,0,1,0,0
2,2026-06-01,052dfc8b-0eb9-4822-8935-0293bd7fc8b8,T-Roc,0,3,0,0
3,2026-06-01,061a7ff6-fd75-423f-b3b9-c423f6b42692,ID.3,0,1,0,0
4,2026-06-01,09e06ecd-e461-4b7f-995d-d628c9cf4c90,ID.3,0,1,0,0
...,...,...,...,...,...,...,...
13892,2026-08-31,f6c38a66-4077-4ed1-a3b8-7063e2c33294,T-Roc,0,1,0,0
13893,2026-08-31,f767ee45-e060-4d0a-9056-933cf896fbf8,T-Roc,0,1,0,0
13894,2026-08-31,f7bb7b4f-fc61-4863-90e9-d841e978dee4,ID.3,0,1,0,0
13895,2026-08-31,fac014e7-f899-4504-aac7-a7dbb99a6786,T-Roc,0,1,0,0


In [19]:
file_path = "modelgpt_output_conversationid_breakdown_q3.xlsx"

mode = "a" if os.path.exists(file_path) else "w"

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode=mode,
    if_sheet_exists="replace" if mode == "a" else None
) as writer:
    intent_summary.to_excel(
        writer,
        sheet_name="intent_summary",
        index=False
    )